# Scarlet planet-cube reproduction

A thin interface to the tested benchmark commands, matching `collaborator_reproduction.ipynb` for the two-galaxy cube.

`majo_planet_cube.fits` is a different problem from `morphology_galaxy_cube_004.fits` in three ways that change the whole setup:

1. **Both sources are intrinsic point sources** (`MORPHSIG=1e-9`) -- a 5750 K G star and a 1000 K brown dwarf, 16 px (1.6") apart at a 5000:1 flux ratio. Scarlet's own `PointSource` is the matching model, not `ExtendedSource`.
2. **The cube is noiseless** -- 0 negative pixels in 2.25M. There is no signal-dependent variance to measure, so the weights are uniform, and chi-square is not the figure of merit; model residual is.
3. **With morphologies pinned at the PSF the model is linear in the spectra**, so there is no bilinear mixing degeneracy. None of the mixing-envelope or multi-start machinery the galaxy campaign needs applies here, and there is only one run rather than a declared A/B/C set.

The PSF is the one thing that has to be exactly right. The simulator rendered sources at a **+0.5 detector pixel offset**, so the kernel is the 4x-oversampled `OVERSAMP` plane rolled by two oversampled cells on both axes and then 4x4 binned -- an integer roll, so no interpolation enters. `DET_SAMP` as shipped, or any centroid-based recentring, leaves a ~50 percent model residual at short wavelengths that shrinks as the PSF broadens.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

# Configuration: edit only this cell. Run the notebook from the Scarlet checkout.
SCARLET_REPO = Path.cwd().resolve()
DATA_ROOT = Path('/path/to/collaborator/data')
OUTPUT_ROOT = SCARLET_REPO / 'benchmark_artifacts' / 'planet'
MAX_ITER = 1500

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
print('repository', SCARLET_REPO)
print('data root ', DATA_ROOT)
print('outputs   ', OUTPUT_ROOT)

In [ ]:
fit_command = [
    sys.executable, '-m', 'benchmarks.run_planet_reproduction',
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
    '--kernel-size', '47',
    '--max-iter', str(MAX_ITER),
    '--relative-tolerance', '1e-9',
]
print(' '.join(fit_command))

The next cell performs the fit and writes the NPZ product and the JSON report. On a cluster, `submit_planet_reproduction.sh` runs the same configuration as a batch job.

Watch the reported iteration count against `MAX_ITER`. A run that stops at the cap has not reached `e_rel`, and per the benchmark reporting rules that is consistent-but-unconverged, not a converged result.

In [ ]:
subprocess.run(fit_command, cwd=SCARLET_REPO, check=True)

In [ ]:
product = OUTPUT_ROOT / 'scarlet_planet_recovery.npz'
report_path = OUTPUT_ROOT / 'scarlet_planet_report.json'

plot_command = [
    sys.executable, '-m', 'benchmarks.plot_planet_reproduction',
    '--product', str(product),
    '--data-root', str(DATA_ROOT),
    '--output-dir', str(OUTPUT_ROOT),
]
subprocess.run(plot_command, cwd=SCARLET_REPO, check=True)

In [ ]:
report = json.loads(report_path.read_text())
print('iterations', report['iterations'], 'of', MAX_ITER,
      '(capped -- not converged)' if report['iterations'] >= MAX_ITER else '(converged)')
for name, scores in report['sources'].items():
    print(f"  {name:12s} relative L2 {scores['relative_l2']:.4e}"
          f"   integrated flux ratio {scores['integrated_flux_ratio']:.6f}")

A full-vector spectral score is flux weighted and hides weak wavelength intervals, so the spectral figure also shows the fractional error per channel. For the brown dwarf that panel excludes the channels below 1e-4 of its peak: shortward of ~0.7 um the truth companion flux is essentially zero, and a ratio there measures nothing.

The recovered spectra should be compared against the fixed-morphology linear solve in the `lisasep` repository (`examples/benchmark_planet/fit_planet_fixed.py`), which on noiseless data is exact. Agreement between the two is a forward-model correctness check -- it says the PSF phase and the geometry are right. It is **not** a sensitivity result: whether a 5000:1 companion is recoverable in practice needs noise injected.

In [ ]:
from IPython.display import Image, Markdown, display

for name in (
    'scarlet_planet_spectra.png',
    'scarlet_planet_morphologies.png',
    'scarlet_planet_residual.png',
):
    path = OUTPUT_ROOT / name
    if path.exists():
        display(Markdown(f'### {name}'))
        display(Image(filename=str(path)))